**Unit 1 Assignment: The Model Benchmark Challenge**

Objective: In this assignment, you will step beyond simply using a model and instead evaluate the architectural differences between BERT, RoBERTa, and BART. You will force these models to perform tasks they might not be designed for, to observe why architecture matters.

In [1]:
!pip install transformers nltk


In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [3]:
from transformers import pipeline, set_seed
set_seed(42)


In [5]:
models = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "BART": "facebook/bart-base"
}


**Experiment 1: Text Generation**



---



We use the text-generation pipeline to test whether different models can generate new text from a prompt. This experiment demonstrates that decoder or encoder-decoder models can generate sequences, while encoder-only models fail.

In [6]:
print("=== TEXT GENERATION ===\n")

for name, model in models.items():
    print(f"Model: {name}")
    try:
        generator = pipeline("text-generation", model=model)
        output = generator("The future of Artificial Intelligence is", max_new_tokens=40)
        print(output[0]["generated_text"])
    except Exception as e:
        print("Failed:", e)
    print("-" * 80)



=== TEXT GENERATION ===

Model: BERT


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


The future of Artificial Intelligence is liking Thunder evasionnumber Sega flownzai unfocusedRangeICICnumbernumber unfocusedRangeazyEvidenceEvidence trendyEvidence varying 283EvidenceEvidence varying varying oun unfocusedRangeAcknowledEvidence MajestyEvidence oriented pinpointAcknowled varyingEvidence Msconf trendy cath
--------------------------------------------------------------------------------


**Experiment 2: Fill-Mask**



---



We use the fill-mask pipeline to predict a missing word in a sentence. This task aligns with how BERT and RoBERTa are trained, making them suitable for masked language modeling.

In [7]:
print("\n=== FILL MASK ===\n")

sentence = "The goal of Generative AI is to [MASK] new content."

for name, model in models.items():
    print(f"Model: {name}")
    try:
        fill_mask = pipeline("fill-mask", model=model)
        results = fill_mask(sentence)
        for r in results[:3]:
            print(r["sequence"], " | score:", round(r["score"], 3))
    except Exception as e:
        print("Failed:", e)
    print("-" * 80)


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



=== FILL MASK ===

Model: BERT


Device set to use cpu


the goal of generative ai is to create new content.  | score: 0.54
the goal of generative ai is to generate new content.  | score: 0.156
the goal of generative ai is to produce new content.  | score: 0.054
--------------------------------------------------------------------------------
Model: RoBERTa


Device set to use cpu


Failed: No mask_token (<mask>) found on the input
--------------------------------------------------------------------------------
Model: BART


Device set to use cpu


Failed: No mask_token (<mask>) found on the input
--------------------------------------------------------------------------------


**Experiment 3: Question Answering**



---



We use the question-answering pipeline to extract an answer from a given context. This experiment shows that models not fine-tuned for QA often produce incomplete or incorrect answers.

In [8]:
print("\n=== QUESTION ANSWERING ===\n")

context = "Generative AI poses significant risks such as hallucinations, bias, and deepfakes."
question = "What are the risks?"

for name, model in models.items():
    print(f"Model: {name}")
    try:
        qa = pipeline("question-answering", model=model)
        result = qa(question=question, context=context)
        print("Answer:", result["answer"])
    except Exception as e:
        print("Failed:", e)
    print("-" * 80)



=== QUESTION ANSWERING ===

Model: BERT


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


Answer: , and deepfakes
--------------------------------------------------------------------------------
Model: RoBERTa


Some weights of RobertaForQuestionAnswering were not initialized from the model checkpoint at roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu
Some weights of BartForQuestionAnswering were not initialized from the model checkpoint at facebook/bart-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Answer: poses significant risks such as hallucinations, bias, and deepfakes
--------------------------------------------------------------------------------
Model: BART


Device set to use cpu


Answer: significant risks such as
--------------------------------------------------------------------------------


**Deliverable: Observation Table**



---

| Task       | Model   | Classification (Success/Failure) | Observation (What actually happened?)                           | Why did this happen? (Architectural Reason)                                       |
| ---------- | ------- | -------------------------------- | --------------------------------------------------------------- | --------------------------------------------------------------------------------- |
| Generation | BERT    | Failure                          | Generated only dots, no meaningful continuation.                | BERT is an Encoder-only model and lacks autoregressive next-token prediction.     |
| Generation | RoBERTa | Failure                          | Did not generate any continuation.                              | RoBERTa is also Encoder-only and not trained for text generation.                 |
| Generation | BART    | Failure                          | Generated random and incoherent words.                          | BART was loaded as causal LM without fine-tuning, leading to unstable generation. |
| Fill-Mask  | BERT    | Success                          | Correctly predicted words like "create", "generate", "produce". | BERT is trained using Masked Language Modeling (MLM).                             |
| Fill-Mask  | RoBERTa | Failure                          | Failed due to missing `<mask>` token.                           | RoBERTa uses `<mask>` token instead of `[MASK]`.                                  |
| Fill-Mask  | BART    | Failure                          | Failed due to missing `<mask>` token.                           | BART also uses `<mask>` token, not `[MASK]`.                                      |
| QA         | BERT    | Partial Success                  | Returned incomplete answer: ", and deepfakes".                  | Base BERT is not fine-tuned for QA.                                               |
| QA         | RoBERTa | Success                          | Returned full correct answer.                                   | Even without fine-tuning, encoder models can extract spans.                       |
| QA         | BART    | Partial Success                  | Returned partial phrase: "significant risks such as".           | BART not fine-tuned for extractive QA.                                            |

